# Week 2 — Talking to a raw model on the ALCF inference endpoint

**AI for Theological Inquiry**

Today we open the hood on the language model itself. You'll:

1. Say **hello** to an open-weight model (one call).
2. Turn the **temperature** knob and watch determinism vs. sampling.
3. Provoke a **hallucination** on purpose — the money moment.
4. Fix it by **grounding** the model in real text — RAG, by hand.

> **Before you run anything**, complete the handout steps: `pip install openai`, download `inference_auth_token.py`, and run `python inference_auth_token.py authenticate`. Keep this notebook in the same folder as `inference_auth_token.py`.

## Setup

We build one OpenAI-compatible client and a tiny `ask()` helper we reuse below.
If `list_models()` shows a different chat model than the default, paste that exact ID into `MODEL`.

In [ ]:
from openai import OpenAI
from inference_auth_token import get_access_token

# The ALCF endpoint is OpenAI-compatible. Sophia / vLLM base URL:
BASE_URL = "https://inference-api.alcf.anl.gov/resource_server/sophia/vllm/v1"

# Use an exact ID from list_models() below. The served set changes over time.
MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"

client = OpenAI(api_key=get_access_token(), base_url=BASE_URL)

def ask(prompt, temperature=0, model=None, system=None):
    """Send one chat message and return the model's text."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = client.chat.completions.create(
        model=model or MODEL,
        messages=messages,
        temperature=temperature,
    )
    return resp.choices[0].message.content

### (optional) Which models are being served right now?

Handy to run once — copy a current chat-model ID into `MODEL` above if the default is rejected.

In [ ]:
import requests

def list_models():
    url = "https://inference-api.alcf.anl.gov/resource_server/list-endpoints"
    r = requests.get(url, headers={"Authorization": f"Bearer {get_access_token()}"})
    r.raise_for_status()
    return r.json()

list_models()

## 0. Hello — one call to a raw model

No agent, no tools — just `messages in -> completion out`.

In [ ]:
print(ask("What does Romans 7:24 say?", temperature=0))

## 1. Temperature — the same box, one knob

`temperature=0` takes the most-likely token every time (**deterministic**).
High temperature **samples** more widely (**creative, unstable**).

Run the cell and compare: the two `temp=0` answers should be identical; the three
`temp=1.5` answers should all differ.

In [ ]:
prompt = "In one vivid sentence, describe the parable of the lost sheep."

print("=== temperature = 0 (deterministic) ===")
for i in range(2):
    print(f"[run {i+1}] {ask(prompt, temperature=0)}\n")

print("=== temperature = 1.5 (sampled) ===")
for i in range(3):
    print(f"[run {i+1}] {ask(prompt, temperature=1.5)}\n")

## 2. Hallucination — confident, fluent, and wrong

There is **no Book of Hezekiah** in the biblical canon. Ask anyway.
A small model will often *invent* a verse in flawless cadence — because it is
completing a **pattern** ("a verse citation"), not looking anything up.

Try a few provocations. Not every model falls for every one; that's part of the lesson.

In [ ]:
provocations = [
    "Quote Hezekiah 3:16 and briefly explain it.",
    "What does 2 Ecclesiastes 5:9 teach about wealth?",
    "Summarize the Epistle of Paul to the Corinthians III.",
]

for p in provocations:
    print(f"PROMPT: {p}")
    print(ask(p, temperature=0))
    print("-" * 70)

> **Save this as your disclosure artifact.** Copy one clear hallucination into the
> cell below — the model ID, the temperature, the prompt, and the wrong output.
> This is the first entry in your project's AI-use appendix.


In [ ]:
artifact = f"""
MODEL:       {MODEL}
TEMPERATURE: 0
PROMPT:      Quote Hezekiah 3:16 and briefly explain it.
OUTPUT:      <paste the fabricated verse the model produced here>
NOTE:        There is no Book of Hezekiah. The model fabricated a plausible verse.
"""
print(artifact)
# Optionally save it:
# open("my_first_hallucination.txt", "w").write(artifact)

## 3. Grounding by hand — RAG in one prompt

The fix for hallucination is not a bigger model — it's **giving the model the real
text** and telling it to use *only* that. Below we paste an actual passage and
constrain the answer. If the fact isn't in the text, a well-grounded model says so.

Here *you* choose the passage — you're doing the **retrieve** step by hand. In §4
just below, the code will do that step for you automatically. That's RAG.

In [ ]:
# A real passage (Romans 7:24-25, ESV-style paraphrase for the demo).
context = """
Wretched man that I am! Who will deliver me from this body of death?
Thanks be to God through Jesus Christ our Lord! So then, I myself serve
the law of God with my mind, but with my flesh I serve the law of sin.
"""

system = ("You are a careful study assistant. Answer ONLY using the passage "
          "provided. If the answer is not in the passage, reply exactly: "
          "'Not found in the provided text.'")

grounded_prompt = f"Passage:\n{context}\n\nQuestion: According to this passage, who delivers the speaker from the body of death?"
print("GROUNDED (fact is present):")
print(ask(grounded_prompt, temperature=0, system=system))
print("-" * 70)

absent_prompt = f"Passage:\n{context}\n\nQuestion: What year was the Epistle to the Romans written?"
print("GROUNDED (fact is absent -> should refuse):")
print(ask(absent_prompt, temperature=0, system=system))

## 4. Real RAG — let the model *retrieve*, then ground

In §3 you did the **retrieve** step by hand: *you* already knew which passage to paste in.
A **RAG** system (Retrieval-Augmented Generation) does that step *for you*:

1. **Index** — turn each passage into an embedding vector (*meaning as geometry* — Slide 5).
2. **Retrieve** — embed the *question*, find the nearest passages by cosine similarity.
3. **Ground** — paste only those passages into the prompt and answer from them (that's §3).

It's the same OpenAI-compatible client — we just call `/embeddings` instead of `/chat`.
This six-verse corpus stands in for "your Week-4 text collection."

In [ ]:
import math

# A tiny "library" — six passages on clearly different themes.
CORPUS = [
    {"ref": "Romans 7:24-25",
     "text": "Wretched man that I am! Who will deliver me from this body of death? "
             "Thanks be to God through Jesus Christ our Lord!"},
    {"ref": "Psalm 23:1-2",
     "text": "The Lord is my shepherd; I shall not want. He makes me lie down in "
             "green pastures. He leads me beside still waters."},
    {"ref": "Luke 15:4 (the lost sheep)",
     "text": "What man of you, having a hundred sheep, if he has lost one of them, "
             "does not leave the ninety-nine and go after the one that is lost?"},
    {"ref": "1 Corinthians 13:4",
     "text": "Love is patient and kind; love does not envy or boast; it is not "
             "arrogant or rude."},
    {"ref": "Micah 6:8",
     "text": "What does the Lord require of you but to do justice, to love kindness, "
             "and to walk humbly with your God?"},
    {"ref": "Genesis 1:1",
     "text": "In the beginning, God created the heavens and the earth."},
]

# An embedding model served on ALCF. Like the chat model, the served set changes:
# check list_models() for an id containing "embed"/"bge"/"nomic" and paste it here.
EMBED_MODEL = "nomic-ai/nomic-embed-text-v1.5"

def embed(texts):
    """Embed a list of strings via the SAME client — OpenAI /embeddings path."""
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [d.embedding for d in resp.data]

def cosine(a, b):
    """Cosine similarity of two vectors — 'how close in meaning' (Slide 5)."""
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(y * y for y in b))
    return dot / (na * nb + 1e-9)

# Build the index once. If no embedding model is served, fall back to a simple
# keyword score so the lesson still runs — flip nothing, it's automatic.
USE_EMBEDDINGS = True
try:
    doc_vectors = embed([d["text"] for d in CORPUS])
    print(f"Indexed {len(CORPUS)} passages with embeddings ({EMBED_MODEL}).")
except Exception as e:
    USE_EMBEDDINGS = False
    print(f"[embeddings unavailable: {e}]\nFalling back to keyword overlap.")

In [ ]:
def _lex(query, text):
    """Keyword-overlap fallback score (used only if embeddings aren't served)."""
    q, t = set(query.lower().split()), set(text.lower().split())
    return len(q & t) / (len(q) + 1e-9)

def retrieve(query, k=2, show=True):
    """Return the k passages closest to the query (the 'R' in RAG)."""
    if USE_EMBEDDINGS:
        qv = embed([query])[0]
        scored = [(cosine(qv, dv), d) for dv, d in zip(doc_vectors, CORPUS)]
    else:
        scored = [(_lex(query, d["text"]), d) for d in CORPUS]
    scored.sort(key=lambda s: s[0], reverse=True)
    if show:
        print(f"Query: {query}\nRanked passages (similarity):")
        for score, d in scored:
            print(f"  {score:5.3f}  {d['ref']}")
        print("-" * 70)
    return [d for _, d in scored[:k]]

# Watch the geometry pick the right text — nothing was hard-coded:
_ = retrieve("Who rescues me from this body of death?", k=2)
_ = retrieve("What does God ask of us — how should we live?", k=2)

In [ ]:
RAG_SYSTEM = ("You are a careful study assistant. Answer ONLY using the passages "
              "provided, and cite the reference(s) you used. If the answer is not "
              "in them, reply exactly: 'Not found in the provided text.'")

def rag_answer(query, k=2):
    """Full RAG: retrieve the k best passages, then answer grounded in them."""
    hits = retrieve(query, k=k, show=False)
    context = "\n\n".join(f"[{d['ref']}] {d['text']}" for d in hits)
    prompt = f"Passages:\n{context}\n\nQuestion: {query}"
    print(f"RETRIEVED: {', '.join(d['ref'] for d in hits)}")
    print(ask(prompt, temperature=0, system=RAG_SYSTEM))
    print("=" * 70)

# Fact IS in the corpus -> grounded, cited answer:
rag_answer("Who delivers the speaker from the body of death?")

# Fact is NOT in the corpus -> retrieval still returns its closest guess,
# but grounding makes the model refuse instead of hallucinate:
rag_answer("In what year was the Epistle to the Romans written?")

### That's RAG.

Thirty lines: **index → retrieve → ground**. The same three steps power every
"chat with your documents" product — the difference is scale (thousands of
passages, a real vector database) and polish, not the idea.

Look at the two questions you just ran:

- **Answer in the corpus** → retrieval finds the right passage *by meaning*
  (not keyword matching) and the model answers with a citation.
- **Answer absent** → retrieval still hands back its closest guess, but the
  grounding instruction makes the model *decline* rather than invent — the exact
  cure for the §2 hallucination.

**In Week 4** you swap these six verses for *your* corpus, and the pure-Python
`cosine` for a real vector database — but the shape you just built does not change.

> **Disclosure note:** a RAG answer depends on *what was retrieved*. This course's
> standard is to record the retrieved reference(s) alongside the model ID and
> temperature — so a reader can check the sources, not just the answer.

## Project touchpoint — point the model at *your* question

Take the interest statement you wrote in Week 1. Turn it into **one factual
question** an 8B model could plausibly get wrong (a specific date, citation,
attribution, or minority-tradition claim). Run it at `temp=0`, then **fact-check
the answer yourself** and note where it was fluent-but-wrong.

**Keep that failing question** — it becomes a seed for your Week 6 evaluation set.

In [ ]:
my_question = "<< write one factual question from your own research interest >>"

# print(ask(my_question, temperature=0))